# Team-Fit Backtest — the referee

This notebook is the **scoring harness** for all later team-fit work. It contains **no fit-scoring logic** — it only *grades* a fit score. A fit score is any callable `fit_score(player_id, from_season, to_team) -> float | None`.

**The question:** does a fit score predict what actually happened after real team changes, better than two dumb baselines?

**Pieces** (all logic in `analysis/team_fit_backtest.py`, backed by SQL views):
1. **Cohort** `v_team_fit_cohort` — players who changed primary team between consecutive seasons, ≥500 on-court offensive possessions in both.
2. **Outcomes** `team_fit_outcomes` (materialized from `v_team_fit_outcomes`) — post-move deltas: on/off net (`mv_player_onoff.net_swing`) and TS% (`v_player_usage_efficiency.ts_pct`), with **age** + **prior-season minutes** as controls.
3. **Report** `score_report(fit_score, ...)` — Spearman rank corr + partial corr (controls removed) + **lift vs two baselines**.
4. **Baselines (the bar):** (a) good player × good team, (b) style-similarity-only.

Data is all tables/matviews/scalar-jsonb views — **nothing scans `shot_event`** — but the joined outcome view was still too slow for the PostgREST timeout, so it's **materialized** into `team_fit_outcomes` (rebuild when seasons update; see the module docstring).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[0] / 'analysis') if Path.cwd().name != 'analysis' else str(Path.cwd()))
import team_fit_backtest as tfb

data = tfb.load_data()   # (outcomes, names, netrtg, vector)
outcomes, names, netrtg, vector = data
print('cohort moves loaded:', len(outcomes))

## 1. Print the bar
The baseline numbers every future fit score must beat. (Each baseline is scored as the *candidate*, so its self-lift is 0 — the number that matters is its raw + partial Spearman vs `delta_net`.)

In [ ]:
tfb.print_the_bar(data=data)

## 2. Grade your own fit score
Pass any `fit_score(player_id, from_season, to_team)` to `score_report`. Example below uses a deterministic dummy (hash-based) score, which should land near zero correlation — i.e. no signal, as expected for noise. Replace it with a real fit score later.

In [ ]:
def dummy_fit_score(player_id, from_season, to_team):
    # deterministic pseudo-random in [0,1); a real score goes here later
    return ((player_id * 2654435761) % 10_000) / 10_000.0

_ = tfb.score_report(dummy_fit_score, outcome_col='delta_net', label='dummy', data=data)

## Notes / caveats for the next discussion
- **The bar is low.** Both baselines sit near zero Spearman vs `delta_net` (good-player×good-team is even slightly *negative*). That's partly real (naive signals barely predict post-move impact change) and partly the **outcome metric**: on/off net is *baseline-relative* (resets each team/season) and noisy, so `delta_net` punishes regression-to-mean. A future iteration may want a better outcome (e.g. role-adjusted impact, or a team-quality-residualized target).
- **`delta_ts`** is cleaner (TS% is absolute, not baseline-relative) but a narrower slice of “fit.”
- **Controls** (age, prior minutes) are residualized out in the `partial` column — a fit score must beat them, not ride them.
- **2024-25→2025-26** uses the in-progress current season; the ≥500 off-poss filter keeps it to players with real minutes, but treat it as provisional.
- **`mv_player_onoff` dedup bug** surfaced: it carries a spurious low-poss row alongside the real one per (player, season, team); the outcome view dedups by max `poss_on`. Worth fixing upstream.